##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model.
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
- Validate the model on 3 YouTube videos, each clearly showing one of your three chosen action classes.

---

**Student:** Alhanouf Alqhatani  
**Selected classes:** basketball, biking, tennis_swing

### 1. Install dependencies

Skip this cell if the packages are already in your environment.

In [ ]:
!pip install -q opencv-python-headless matplotlib scikit-learn tensorflow yt-dlp

### 2. Imports

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

### 3. Reproducibility

In [ ]:
# Set seed for consistent results
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

print(f"Seed set to: {seed_constant}")

### 4. Locate the UCF11 dataset

Download UCF11 (`UCF11_updated_mpg.rar`) from <https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php>, extract it next to this notebook, and verify the folder name below.

In [ ]:
# Path to the extracted UCF11 dataset folder.
# After extracting UCF11_updated_mpg.rar you typically get a folder named 'action_youtube_naudio'.
DATASET_DIR = os.path.join(os.getcwd(), 'action_youtube_naudio')

all_classes = [cls for cls in os.listdir(DATASET_DIR) if not cls.startswith('.')]

print("=" * 50)
print("Available Classes in Dataset:")
for cls in all_classes:
    print(f"  - {cls}")
print("=" * 50)

### 5. Preprocess the videos

Pick three classes, extract frames (resized + normalised), and build the labelled dataset.

In [ ]:
CLASSES_LIST = ["basketball", "biking", "tennis_swing"]
print(f"Selected Classes: {CLASSES_LIST}")

image_height, image_width = 64, 64
max_images_per_class = 8000
model_output_size = len(CLASSES_LIST)

def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    while True:
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, (image_height, image_width))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)
    video_reader.release()
    return frames_list

def create_dataset():
    temp_features = []
    features = []
    labels = []
    class_frame_counts = {}

    for class_index, class_name in enumerate(CLASSES_LIST):
        class_path = os.path.join(DATASET_DIR, class_name)

        if not os.path.exists(class_path):
            print(f"Warning: {class_name} folder missing!")
            continue

        for root, dirs, files in os.walk(class_path):
            for file_name in files:
                if file_name.lower().endswith((".mpg", ".avi", ".mp4", ".mov")):
                    video_file_path = os.path.join(root, file_name)
                    frames = frames_extraction(video_file_path)
                    temp_features.extend(frames)

        class_frame_counts[class_name] = len(temp_features)

        if len(temp_features) > 0:
            sample_size = min(len(temp_features), max_images_per_class)
            features.extend(random.sample(temp_features, sample_size))
            labels.extend([class_index] * sample_size)
            temp_features.clear()

    # Summary Table
    print("=" * 45)
    print(f"{'Class':<25} {'Total Frames':>12} {'Sampled':>6}")
    print("=" * 45)
    for cls, count in class_frame_counts.items():
        sampled = min(count, max_images_per_class)
        print(f"{cls:<25} {count:>12,} {sampled:>6,}")
    print("=" * 45)

    return np.asarray(features), np.array(labels)

# Process the dataset
features, labels = create_dataset()
if len(labels) > 0:
    one_hot_encoded_labels = to_categorical(labels)
else:
    print("Error: No frames extracted. Please check the video files inside folders.")

### 6. Split the dataset (80% train / 20% test)

In [ ]:
features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels, test_size=0.2, shuffle=True, random_state=seed_constant)

print(f"Training samples : {len(features_train):,}")
print(f"Testing samples  : {len(features_test):,}\n")

### 7. Build the CNN model

In [ ]:
def create_model():
    """Builds a CNN model for image-level action recognition."""
    model = Sequential()

    # Feature Extraction
    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu', input_shape=(image_height, image_width, 3)))
    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Global average pooling to reduce dimensionality
    model.add(GlobalAveragePooling2D())

    # Fully Connected Layers
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())

    # Output Layer: size matches the number of selected classes
    model.add(Dense(model_output_size, activation='softmax'))

    model.summary()
    return model

# Initialize the model
model = create_model()

### 8. Train the model

In [ ]:
# Define early stopping to prevent overfitting
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    restore_best_weights=True
)

# Compile the model with Adam optimizer
model.compile(
    loss='categorical_crossentropy',
    optimizer='Adam',
    metrics=['accuracy']
)

# Start training
model_training_history = model.fit(
    x=features_train,
    y=labels_train,
    epochs=30,
    batch_size=4,
    shuffle=True,
    validation_split=0.2,
    callbacks=[early_stopping_callback]
)

### 9. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Alhanouf Alqhatani — Action Recognition Training Results', fontsize=14, fontweight='bold')

# Accuracy Plot
axes[0].plot(model_training_history.history['accuracy'], label='Train Accuracy', color='steelblue', linewidth=2)
axes[0].plot(model_training_history.history['val_accuracy'], label='Val Accuracy', color='orange', linewidth=2)
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Loss Plot
axes[1].plot(model_training_history.history['loss'], label='Train Loss', color='steelblue', linewidth=2)
axes[1].plot(model_training_history.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('Alhanouf_Alqhatani_training_plot.png', dpi=150)
plt.show()

### 10. Evaluate on the held-out test set

In [ ]:
loss, accuracy = model.evaluate(features_test, labels_test)

print("\n" + "=" * 40)
print("         TEST EVALUATION RESULTS")
print('        Alhanouf Alqhatani — ARTI560')
print("=" * 40)

results_df = pd.DataFrame({
    'Metric': ['Test Loss', 'Test Accuracy'],
    'Value': [f"{loss:.4f}", f"{accuracy * 100:.2f}%"]
})

print(results_df.to_string(index=False))
print("=" * 40)

### 11. Save the trained model

In [ ]:
student_name = "Alhanouf_Alqhatani"
save_path = f"{student_name}_action_model.h5"

model.save(save_path)

print(f"\nModel saved as: {save_path}\n")

### 12. Load the model back

Sanity-check that the saved model file is valid and reloads correctly.

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('Alhanouf_Alqhatani_action_model.h5')

### 13. Validate on three external YouTube videos

Place three short videos (one per class) next to this notebook: `basketball.mp4`, `biking.mp4`, `tennis_swing.mp4`. They can be downloaded with `yt-dlp`, e.g.:

```bash
yt-dlp -f mp4 -o "basketball.mp4" "<youtube_url>"
yt-dlp -f mp4 -o "biking.mp4" "<youtube_url>"
yt-dlp -f mp4 -o "tennis_swing.mp4" "<youtube_url>"
```

In [ ]:
test_results = []

def predict_video(video_path):
    try:
        frames = frames_extraction(video_path)
        if frames is None or len(frames) == 0:
            return None

        target_frame = frames[len(frames) // 2]
        input_data = np.expand_dims(target_frame, axis=0)

        prediction = model.predict(input_data, verbose=0)
        predicted_class_index = np.argmax(prediction)
        confidence = prediction[0][predicted_class_index]
        predicted_class_name = CLASSES_LIST[predicted_class_index]
        return predicted_class_name, confidence
    except Exception as e:
        print(f"Technical Error Details: {e}")
        return None

# --- Test Section ---
local_test_videos = {
    'basketball':   os.path.join(os.getcwd(), 'basketball.mp4'),
    'biking':       os.path.join(os.getcwd(), 'biking.mp4'),
    'tennis_swing': os.path.join(os.getcwd(), 'tennis_swing.mp4'),
}

print("=" * 60)
print(f"{'Video File':<25} {'Predicted Action':<22} {'Confidence':>10}")
print("=" * 60)

for original_class, video_path in local_test_videos.items():
    if os.path.exists(video_path):
        result = predict_video(video_path)
        if result:
            predicted_class, confidence = result
            print(f"{original_class:<25} {predicted_class:<22} {confidence*100:>9.2f}%")
            test_results.append({
                'predicted': f"{original_class} ({predicted_class})",
                'confidence': confidence * 100,
            })
        else:
            print(f"{original_class:<25} {'Error':<22} {'0.00%':>10}")
    else:
        print(f"{original_class:<25} {'File Not Found':<22} {'N/A':>10}")

print("=" * 60)

### 14. Plot prediction confidences

In [ ]:
if test_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#4682B4', '#FF8C00', '#2E8B57']

    classes = [r['predicted'] for r in test_results]
    confidences = [r['confidence'] for r in test_results]

    bars = ax.barh(classes, confidences, color=colors[:len(test_results)], edgecolor='white', height=0.6)

    for bar in bars:
        width = bar.get_width()
        ax.text(width - 5, bar.get_y() + bar.get_height()/2,
                f'{width:.2f}%',
                va='center', ha='right', color='white',
                fontweight='bold', fontsize=12)

    ax.set_xlim(0, 100)
    ax.set_xlabel('Confidence (%)', fontsize=12)
    ax.set_title('Alhanouf Alqhatani — Prediction Confidence per Class', fontweight='bold', fontsize=14)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('Alhanouf_Alqhatani_predictions.png', dpi=150)
    plt.show()
else:
    print("No prediction results to plot. Please run the table cell first.")

### Summary

Once this notebook finishes running you should have these deliverables in `lab07-action-recognition/`:

- `Alhanouf_Alqhatani_action_model.h5`
- `Alhanouf_Alqhatani_training_plot.png`
- `Alhanouf_Alqhatani_predictions.png`
- `basketball.mp4`, `biking.mp4`, `tennis_swing.mp4` (validation videos)

Push all of them to the GitHub repo alongside this notebook.